# Questão 4 — PRISM na base da Questão 3

Base usada: a mesma **Car Evaluation** da Questão 3, pra poder comparar com as árvores.

## 1. Base de dados

Mesmo dataset e mesmo split treino/teste (80/20 estratificado) da Questão 3, só que
lido direto da pasta `Questao_3/dados`.

In [1]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append("Algoritimos")
from prism import prism

DATA_PATH = os.path.join("..", "Questao_3", "dados", "car_evaluation.csv")
df = pd.read_csv(DATA_PATH)
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [2]:
train, test = train_test_split(df, test_size=0.2, stratify=df["class"], random_state=42)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
len(train), len(test)

(1382, 346)

## 2. Como o PRISM escolhe cada condição

Trecho de `prism.py`: para cada valor de cada atributo ainda não usado na regra, calcula
`p/t` (proporção de exemplos da classe alvo entre os que têm aquele valor) e fica com o
melhor. Isso se repete até a regra cobrir só exemplos de uma classe.

```python
best = None
for attr in available:
    for value in subgroup[attr].unique():
        slice_ = subgroup[subgroup[attr] == value]
        t = len(slice_)
        p = (slice_[self.target] == class_label).sum()
        score = p / t
        if best is None or score > best[0] or (score == best[0] and p > best[1]):
            best = (score, p, attr, value)
```

Depois de cravar a regra, os exemplos cobertos por ela saem do conjunto pendente e o
processo recomeça pra formar a próxima regra, até esgotar os exemplos daquela classe.

## 3. Rodando o PRISM

In [3]:
model = prism(train, verbose=True)
model.fit("class")
len(model.rules)

202

In [4]:
model.export_log(os.path.join("outputs", "prism_log.txt"))
rules_text = model.export_rules(os.path.join("outputs", "prism_regras.txt"))
print(rules_text[:800])

SE persons = 4 E safety = high E buying = high E maint = med ENTAO class = acc
SE persons = 4 E safety = high E buying = high E maint = high ENTAO class = acc
SE persons = 4 E safety = high E buying = med E maint = high ENTAO class = acc
SE safety = med E persons = 4 E maint = med E buying = med ENTAO class = acc
SE persons = more E safety = high E buying = high E maint = low E lug_boot = med ENTAO class = acc
SE persons = more E safety = high E buying = high E maint = med E lug_boot = big ENTAO class = acc
SE safety = med E persons = more E buying = med E maint = med E doors = 3 ENTAO class = acc
SE safety = med E persons = more E lug_boot = big E maint = high E buying = med ENTAO class = acc
SE safety = med E persons = 4 E buying = low E maint = high ENTAO class = acc
SE persons = more E


## 4. Avaliação

Mesmas métricas usadas na Questão 3, calculadas no conjunto de teste.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

predictions = model.classify_dataset(df=test, filepath=os.path.join("outputs", "prism_predicoes.csv"))

metrics = {
    "acuracia": accuracy_score(predictions["Real"], predictions["Predito"]),
    "precisao": precision_score(predictions["Real"], predictions["Predito"], average="weighted", zero_division=0),
    "recall": recall_score(predictions["Real"], predictions["Predito"], average="weighted", zero_division=0),
    "f1": f1_score(predictions["Real"], predictions["Predito"], average="weighted", zero_division=0),
}
metrics

{'acuracia': 0.8641618497109826,
 'precisao': 0.8610758261730226,
 'recall': 0.8641618497109826,
 'f1': 0.8586547020279471}

In [6]:
print(classification_report(predictions["Real"], predictions["Predito"], zero_division=0))

              precision    recall  f1-score   support

         acc       0.72      0.78      0.75        77
        good       0.75      0.43      0.55        14
       unacc       0.92      0.94      0.93       242
       vgood       0.71      0.38      0.50        13

    accuracy                           0.86       346
   macro avg       0.78      0.63      0.68       346
weighted avg       0.86      0.86      0.86       346



O PRISM fica um pouco atrás das árvores da Questão 3 (que chegaram perto de 90-91% de
acurácia). Faz sentido: cada regra do PRISM é construída olhando só pra uma classe por
vez e não reaproveita a estrutura entre classes como uma árvore faz, então ele tende a
generalizar um pouco pior nas classes com poucos exemplos (`good`, `vgood`).

## 5. Arquivos gerados

Em `Questao_4/outputs/`: `prism_log.txt` (passo a passo da construção de cada regra),
`prism_regras.txt` (base de regras final) e `prism_predicoes.csv` (predições no teste,
usadas para calcular as métricas acima).